# Food-101 Colab Pro Training Workflow

This notebook uses Colab only as the GPU compute environment. Your local machine remains the source of truth for the repository and Food-101 data, but Colab cannot read those files directly. Stage a copy to Google Drive or unpack a local transfer bundle into the Colab runtime before training.

The final report notebook remains `food101_CNN_final_project.ipynb`; run that after training/evaluation to build the final comparison table.

## 1. Runtime Setup

Use `Runtime -> Change runtime type -> GPU` in Colab. The setup mounts Google Drive, optionally unpacks a local transfer bundle into `/content`, then locates the staged repository before importing `food101_cnn`.

In [ ]:
from __future__ import annotations

import json
import os
import shutil
import subprocess
import sys
import tarfile
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

DRIVE_PROJECT_ROOT = Path("/content/drive/MyDrive/food101-cnn")
DRIVE_DATA_ROOT = DRIVE_PROJECT_ROOT / "data"
DRIVE_OUTPUT_ROOT = DRIVE_PROJECT_ROOT / "outputs"
RUN_ROOT = DRIVE_OUTPUT_ROOT / "runs"

# Colab cannot read files on the local machine. Create this archive locally with:
# python scripts/create_colab_bundle.py
# Upload it to Drive, then set RUN_UNPACK_LOCAL_BUNDLE=True.
RUN_UNPACK_LOCAL_BUNDLE = True
COLAB_BUNDLE_PATH = DRIVE_PROJECT_ROOT / "transfer" / "food101_colab_bundle.tar.gz"
BUNDLE_EXTRACT_ROOT = Path("/content")
BUNDLE_REPOSITORY_ROOT = BUNDLE_EXTRACT_ROOT / "food101-cnn"
OVERWRITE_UNPACKED_BUNDLE = False

# Optional: set this when the staged repository lives somewhere else in Drive/Colab.
# The path must contain pyproject.toml, src/food101_cnn/, and the required configs.
REPOSITORY_ROOT_OVERRIDE = ""

REQUIRED_REPOSITORY_FILES = (
    "pyproject.toml",
    "src/food101_cnn/config.py",
    "src/food101_cnn/models/registry.py",
    "scripts/train.py",
    "scripts/evaluate.py",
    "scripts/export_model.py",
    "configs/baseline_cnn_local.yaml",
    "configs/baseline_cnn_simple.yaml",
    "configs/baseline_cnn_colab.yaml",
    "configs/efficientnet_b0_colab.yaml",
    "configs/resnet50_colab.yaml",
    "configs/convnext_tiny_colab.yaml",
)

for directory in [DRIVE_DATA_ROOT / "raw", DRIVE_DATA_ROOT / "processed", DRIVE_DATA_ROOT / "reports", RUN_ROOT, COLAB_BUNDLE_PATH.parent]:
    directory.mkdir(parents=True, exist_ok=True)

if RUN_UNPACK_LOCAL_BUNDLE:
    if not COLAB_BUNDLE_PATH.is_file():
        raise FileNotFoundError(f"Colab bundle not found: {COLAB_BUNDLE_PATH}")
    if BUNDLE_REPOSITORY_ROOT.exists() and OVERWRITE_UNPACKED_BUNDLE:
        shutil.rmtree(BUNDLE_REPOSITORY_ROOT)
    with tarfile.open(COLAB_BUNDLE_PATH, "r:gz") as archive:
        archive.extractall(BUNDLE_EXTRACT_ROOT)


def missing_repository_files(path: Path) -> list[str]:
    return [relative for relative in REQUIRED_REPOSITORY_FILES if not (path / relative).is_file()]


def is_repository_root(path: Path) -> bool:
    return (path / "pyproject.toml").is_file() and (path / "src" / "food101_cnn").is_dir()


def is_usable_repository_root(path: Path) -> bool:
    return is_repository_root(path) and not missing_repository_files(path)


def repository_candidates() -> list[Path]:
    cwd = Path.cwd().resolve()
    candidates = []
    if REPOSITORY_ROOT_OVERRIDE:
        candidates.append(Path(REPOSITORY_ROOT_OVERRIDE).expanduser())
    candidates.extend([
        DRIVE_PROJECT_ROOT,
        DRIVE_PROJECT_ROOT / "food101-cnn",
        DRIVE_PROJECT_ROOT / "repo",
        DRIVE_PROJECT_ROOT / "repository",
        BUNDLE_REPOSITORY_ROOT,
        cwd,
        *cwd.parents,
        Path("/content/food101-cnn"),
    ])

    unique = []
    seen = set()
    for candidate in candidates:
        resolved = candidate.expanduser().resolve()
        if resolved not in seen:
            unique.append(resolved)
            seen.add(resolved)
    return unique


def find_repository_root() -> Path:
    diagnostics = []
    for candidate in repository_candidates():
        if is_usable_repository_root(candidate):
            return candidate
        if is_repository_root(candidate):
            missing = ", ".join(missing_repository_files(candidate))
            diagnostics.append(f"{candidate} -> missing: {missing}")
        else:
            diagnostics.append(f"{candidate} -> not a repository root")
    checked = "\n".join(diagnostics)
    raise FileNotFoundError(
        "Could not locate a usable Food-101 CNN repository source. Colab cannot "
        "import food101_cnn from the local machine directly. The selected root must "
        "include the updated configs, especially configs/baseline_cnn_local.yaml and "
        "configs/baseline_cnn_simple.yaml. If /content/food101-cnn is stale, set "
        "RUN_UNPACK_LOCAL_BUNDLE=False to use the Drive repo, or set "
        "OVERWRITE_UNPACKED_BUNDLE=True with a fresh bundle. Checked:\n"
        f"{checked}"
    )


PROJECT_ROOT = find_repository_root()
os.chdir(PROJECT_ROOT)

SRC_ROOT = PROJECT_ROOT / "src"
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

existing_pythonpath = os.environ.get("PYTHONPATH", "")
pythonpath_parts = [str(SRC_ROOT), *(part for part in existing_pythonpath.split(os.pathsep) if part)]
os.environ["PYTHONPATH"] = os.pathsep.join(dict.fromkeys(pythonpath_parts))

print({
    "project_root": str(PROJECT_ROOT),
    "drive_project_root": str(DRIVE_PROJECT_ROOT),
    "run_root": str(RUN_ROOT),
    "bundle_unpacked": RUN_UNPACK_LOCAL_BUNDLE,
    "required_configs_present": {
        "baseline_cnn_local": str(PROJECT_ROOT / "configs" / "baseline_cnn_local.yaml"),
        "baseline_cnn_simple": str(PROJECT_ROOT / "configs" / "baseline_cnn_simple.yaml"),
    },
    "pythonpath_first": os.environ["PYTHONPATH"].split(os.pathsep)[0],
})

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Repository Import Check

Editable installation is optional for Colab if dependencies are already available. The setup cell adds `src/` to `PYTHONPATH`, so notebook imports and `scripts/*.py` subprocesses can use the local repository source directly.

In [ ]:
RUN_INSTALL = False

if RUN_INSTALL:
    subprocess.run([sys.executable, "-m", "pip", "install", "-e", ".[dev]"], cwd=PROJECT_ROOT, check=True)

import food101_cnn

print({"food101_cnn_version": food101_cnn.__version__, "module_path": food101_cnn.__file__})

In [ ]:
import torch

def gpu_info() -> dict:
    if not torch.cuda.is_available():
        return {"device": "cpu", "name": "CPU", "memory_gb": 0.0}
    props = torch.cuda.get_device_properties(0)
    return {
        "device": "cuda",
        "name": props.name,
        "memory_gb": round(props.total_memory / 1024**3, 2),
    }

GPU = gpu_info()
GPU

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 3. Execution Flags

Default cells print commands without running long jobs. Use `USE_STAGED_LOCAL_DATA=True` only after the local data has been staged into Colab or Drive; it does not read your Mac directly.

In [ ]:
RUN_DATA_PREP = False
RUN_IMAGE_CACHE = False
RUN_TRAINING = True
RUN_EVALUATION = True
RUN_EXPORT_PT = True

# True means: use a staged/uploaded copy of local data under PROJECT_ROOT/data.
# It does not mean Colab can access the local machine filesystem directly.
USE_STAGED_LOCAL_DATA = True
STAGED_LOCAL_DATA_ROOT_OVERRIDE = ""

RESUME_CHECKPOINTS = {
    # "resnet50": "/content/drive/MyDrive/food101-cnn/outputs/runs/<run>/checkpoints/<run>_best_model.pt",
    #'convnext_tiny': "/content/drive/MyDrive/food101-cnn/outputs/runs/convnext_tiny_20260602-0858_v1/checkpoints/convnext_tiny_20260602-0858_v1_best_model.pt",
}

STAGED_LOCAL_DATA_ROOT = (
    Path(STAGED_LOCAL_DATA_ROOT_OVERRIDE).expanduser().resolve()
    if STAGED_LOCAL_DATA_ROOT_OVERRIDE
    else PROJECT_ROOT / "data"
)
ACTIVE_DATA_ROOT = STAGED_LOCAL_DATA_ROOT if USE_STAGED_LOCAL_DATA else DRIVE_DATA_ROOT
ACTIVE_DATA_SOURCE = "staged_local_copy" if USE_STAGED_LOCAL_DATA else "google_drive"

for directory in [ACTIVE_DATA_ROOT / "raw", ACTIVE_DATA_ROOT / "processed", ACTIVE_DATA_ROOT / "reports"]:
    directory.mkdir(parents=True, exist_ok=True)

INDEX_CSV = ACTIVE_DATA_ROOT / "reports" / "dataset_index.csv"
CACHED_INDEX_CSV = None

print({
    "data_source": ACTIVE_DATA_SOURCE,
    "active_data_root": str(ACTIVE_DATA_ROOT),
    "index_csv": str(INDEX_CSV),
    "run_root": str(RUN_ROOT),
})

## 4. Helpers

All work is executed through repository scripts. This keeps Colab runs compatible with local VS Code usage and the final notebook comparison workflow.

In [ ]:
from food101_cnn.data.dataset import load_dataset_index
from food101_cnn.utils.runs import latest_run_for_model, list_run_manifests, manifest_to_comparison_row


BASELINE_COLAB_EPOCHS = 30
TRANSFER_COLAB_EPOCHS = {
    "efficientnet_b0": 40,
    "resnet50": 35,
    "convnext_tiny": 35,
}
LOCAL_BASELINE_BATCH_SIZE = 8
COLAB_BATCH_SIZE_BY_MEMORY = {
    35: {"baseline_cnn": 96, "efficientnet_b0": 64, "resnet50": 48, "convnext_tiny": 32},
    20: {"baseline_cnn": 80, "efficientnet_b0": 48, "resnet50": 40, "convnext_tiny": 28},
    0: {"baseline_cnn": 64, "efficientnet_b0": 32, "resnet50": 32, "convnext_tiny": 24},
}
MODEL_BATCH_PROFILE = {
    "baseline_cnn_local": "baseline_cnn_local",
    "baseline_cnn_simple": "baseline_cnn",
    "baseline_cnn": "baseline_cnn",
    "efficientnet_b0": "efficientnet_b0",
    "resnet50": "resnet50",
    "convnext_tiny": "convnext_tiny",
}


def command_option(command: list[str], option: str) -> Path | None:
    tokens = [str(item) for item in command]
    if option not in tokens:
        return None
    index = tokens.index(option)
    if index + 1 >= len(tokens):
        return None
    return Path(tokens[index + 1])


def check_index_paths(index_csv: Path, *, sample_size: int = 50) -> None:
    if not index_csv.is_file():
        raise FileNotFoundError(f"Index CSV not found: {index_csv}")
    records = load_dataset_index(index_csv)
    if not records:
        raise ValueError(f"Index CSV has no records: {index_csv}")
    sample = records[:sample_size]
    missing = [record for record in sample if not record.image_path.is_file()]
    if missing:
        first = missing[0]
        if index_csv.name == "cached_index.csv":
            hint = f"Upload/unpack the cached images under {index_csv.parent / 'images'}."
        else:
            hint = (
                f"Upload/unpack raw Food-101 under {ACTIVE_DATA_ROOT / 'raw' / 'food-101'} "
                "or use a processed cached_index.csv from data/processed/<hash>/."
            )
        raise FileNotFoundError(
            "Index points to image files that are not available in this Colab runtime. "
            f"First missing path: {first.image_path}. {hint}"
        )
    print({
        "index_csv": str(index_csv),
        "records": len(records),
        "sample_checked": len(sample),
        "first_image": str(sample[0].image_path),
    })


def run_cli(command: list[str]) -> subprocess.CompletedProcess | None:
    print("$", " ".join(str(item) for item in command))
    if command[0] == "PRINT_ONLY":
        return None

    index_csv = command_option(command, "--index-csv")
    if index_csv is not None:
        check_index_paths(index_csv)

    process = subprocess.Popen(
        [str(item) for item in command],
        cwd=PROJECT_ROOT,
        env=os.environ.copy(),
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    output_tail: list[str] = []
    assert process.stdout is not None
    for line in process.stdout:
        print(line, end="")
        output_tail.append(line)
        output_tail = output_tail[-60:]
    returncode = process.wait()
    if returncode != 0:
        tail_text = "".join(output_tail)
        raise RuntimeError(
            f"Command failed with exit code {returncode}: {' '.join(str(item) for item in command)}\n"
            f"Last command output:\n{tail_text}"
        )
    return subprocess.CompletedProcess(command, returncode)


def adaptive_batch_size(model_name: str) -> int:
    profile = MODEL_BATCH_PROFILE.get(model_name, model_name)
    if profile == "baseline_cnn_local":
        return LOCAL_BASELINE_BATCH_SIZE

    memory = GPU["memory_gb"]
    for threshold in sorted(COLAB_BATCH_SIZE_BY_MEMORY, reverse=True):
        if memory >= threshold:
            table = COLAB_BATCH_SIZE_BY_MEMORY[threshold]
            if profile in table:
                return table[profile]
    supported = ", ".join(sorted(MODEL_BATCH_PROFILE))
    raise KeyError(f"No Colab batch-size profile for {model_name}. Supported: {supported}")


def plan_batch_size(item: dict) -> int:
    if "batch_size" in item:
        return int(item["batch_size"])
    return adaptive_batch_size(str(item["model"]))


def project_config_path(item: dict) -> Path:
    config_path = Path(str(item["config"]))
    if not config_path.is_absolute():
        config_path = PROJECT_ROOT / config_path
    if not config_path.is_file():
        raise FileNotFoundError(
            f"Config for {item['model']} not found: {config_path}. "
            f"Current PROJECT_ROOT is {PROJECT_ROOT}. Rerun the setup cell and verify it points "
            "to the updated Drive repository, not a stale /content copy."
        )
    return config_path


def latest_cached_index() -> Path | None:
    candidates = sorted(
        (ACTIVE_DATA_ROOT / "processed").glob("*/cached_index.csv"),
        key=lambda path: path.stat().st_mtime,
        reverse=True,
    )
    return candidates[0] if candidates else None


def selected_index_csv() -> Path:
    return latest_cached_index() or INDEX_CSV


def train_command(item: dict) -> list[str]:
    model_name = str(item["model"])
    command = [
        sys.executable, "scripts/train.py",
        "--config", str(project_config_path(item)),
        "--index-csv", str(selected_index_csv()),
        "--epochs", str(item["epochs"]),
        "--batch-size", str(plan_batch_size(item)),
        "--device", "cuda" if torch.cuda.is_available() else "cpu",
        "--num-workers", str(item.get("num_workers", 2)),
        "--run-root", str(RUN_ROOT),
        "--log-level", "INFO",
    ]
    if bool(item.get("mixed_precision", True)):
        command.append("--mixed-precision")
    else:
        command.append("--no-mixed-precision")
    if int(item.get("gradient_accumulation_steps", 1)) != 1:
        command.extend(["--gradient-accumulation-steps", str(item["gradient_accumulation_steps"])])
    if model_name in RESUME_CHECKPOINTS:
        command.extend(["--resume-checkpoint", RESUME_CHECKPOINTS[model_name]])
    return command


def evaluate_latest_command(item: dict) -> list[str] | None:
    model_name = str(item["model"])
    manifest = latest_run_for_model(PROJECT_ROOT, model_name, run_root=RUN_ROOT)
    if not manifest:
        print(f"No run manifest found for {model_name}")
        return None
    checkpoint = manifest.get("artifacts", {}).get("checkpoint")
    if not checkpoint:
        print(f"No checkpoint recorded for {model_name}")
        return None
    return [
        sys.executable, "scripts/evaluate.py",
        "--config", str(project_config_path(item)),
        "--checkpoint", checkpoint,
        "--index-csv", str(selected_index_csv()),
        "--split", "test",
        "--batch-size", str(plan_batch_size(item)),
        "--device", "cuda" if torch.cuda.is_available() else "cpu",
        "--num-workers", str(item.get("num_workers", 2)),
        "--run-root", str(RUN_ROOT),
    ]


## 5. Food-101 Data Source

Colab can train only from files visible inside the Colab runtime or mounted Drive. Keep `USE_STAGED_LOCAL_DATA=False` for Drive data. Set it to `True` only after unpacking/uploading your local `data/` directory so it exists under `PROJECT_ROOT/data` or `STAGED_LOCAL_DATA_ROOT_OVERRIDE`.

In [ ]:
DATA_CONFIG = "configs/efficientnet_b0.yaml" if USE_STAGED_LOCAL_DATA else "configs/efficientnet_b0_colab.yaml"

download_command = [
    sys.executable,
    "scripts/download_data.py",
    "--config",
    DATA_CONFIG,
    "--index-output",
    str(INDEX_CSV),
]
if USE_STAGED_LOCAL_DATA:
    download_command.append("--no-download")

data_commands = [
    download_command,
    [
        sys.executable,
        "scripts/validate_images.py",
        "--config",
        DATA_CONFIG,
        "--index-csv",
        str(INDEX_CSV),
        "--report-csv",
        str(ACTIVE_DATA_ROOT / "reports" / "image_validation_report.csv"),
        "--corrupted-output",
        str(ACTIVE_DATA_ROOT / "reports" / "corrupted_images.txt"),
    ],
]

print({
    "active_data_source": ACTIVE_DATA_SOURCE,
    "base_index_exists": INDEX_CSV.is_file(),
    "cached_index": str(latest_cached_index()) if latest_cached_index() else None,
})

if RUN_DATA_PREP:
    for command in data_commands:
        run_cli(command)
else:
    for command in data_commands:
        print(" ".join(str(item) for item in command))

In [ ]:
cache_command = [
    sys.executable,
    "scripts/cache_images.py",
    "--config",
    DATA_CONFIG,
    "--index-csv",
    str(INDEX_CSV),
    "--cache-root",
    str(ACTIVE_DATA_ROOT / "processed"),
]
if RUN_IMAGE_CACHE:
    run_cli(cache_command)
CACHED_INDEX_CSV = latest_cached_index()
print({"cached_index_csv": str(CACHED_INDEX_CSV) if CACHED_INDEX_CSV else None})

## 6. Training Plan

The schedule includes the requested comparison set. `baseline_cnn_local` is intentionally constrained with fewer epochs and a small fixed batch to represent local-machine feasibility. `baseline_cnn_simple` uses the same Colab training schedule and batch policy as `baseline_cnn_colab`; the architecture is the controlled difference.

In [ ]:
TRAINING_PLAN = [
    {
        "model": "baseline_cnn_local",
        "config": "configs/baseline_cnn_local.yaml",
        "epochs": 10,
        "batch_size": LOCAL_BASELINE_BATCH_SIZE,
        "mixed_precision": False,
        "role": "reduced local-style simple CNN baseline",
    },
    {
        "model": "baseline_cnn_simple",
        "config": "configs/baseline_cnn_simple.yaml",
        "epochs": BASELINE_COLAB_EPOCHS,
        "mixed_precision": True,
        "role": "simple CNN architecture with baseline_cnn_colab schedule",
    },
    #{
    #    "model": "baseline_cnn",
    #    "config": "configs/baseline_cnn_colab.yaml",
    #    "epochs": BASELINE_COLAB_EPOCHS,
    #    "mixed_precision": True,
    #    "role": "full custom CNN baseline on Colab schedule",
    #},
    #{
    #    "model": "efficientnet_b0",
    #    "config": "configs/efficientnet_b0_colab.yaml",
    #    "epochs": TRANSFER_COLAB_EPOCHS["efficientnet_b0"],
    #    "mixed_precision": True,
    #    "role": "previous best transfer-learning candidate",
    #},
    #{
    #    "model": "resnet50",
    #    "config": "configs/resnet50_colab.yaml",
    #    "epochs": TRANSFER_COLAB_EPOCHS["resnet50"],
    #    "mixed_precision": True,
    #    "role": "classic transfer-learning comparison",
    #},
    #{
    #    "model": "convnext_tiny",
    #    "config": "configs/convnext_tiny_colab.yaml",
    #    "epochs": TRANSFER_COLAB_EPOCHS["convnext_tiny"],
    #    "mixed_precision": True,
    #    "role": "modern CNN transfer-learning comparison",
    #},
]

import pandas as pd

pd.DataFrame([
    {
        "model": item["model"],
        "config": item["config"],
        "epochs": item["epochs"],
        "batch_size": plan_batch_size(item),
        "mixed_precision": item.get("mixed_precision", True),
        "role": item["role"],
    }
    for item in TRAINING_PLAN
])


In [ ]:
for item in TRAINING_PLAN:
    command = train_command(item)
    if RUN_TRAINING:
        run_cli(command)
    else:
        print(" ".join(str(part) for part in command))


## 7. Evaluation After Training

When `RUN_EVALUATION=True`, each latest run is evaluated on the official test split and its manifest is updated. Final comparison is intentionally left to the final report notebook.

In [ ]:
for item in TRAINING_PLAN:
    command = evaluate_latest_command(item)
    if command is None:
        continue
    if RUN_EVALUATION:
        run_cli(command)
    else:
        print(" ".join(str(part) for part in command))


## 8. Export PyTorch Models

Only native `.pt` export is required here. ONNX is optional and can remain disabled if the dependency is unavailable.

In [ ]:
for item in TRAINING_PLAN:
    manifest = latest_run_for_model(PROJECT_ROOT, item["model"], run_root=RUN_ROOT)
    checkpoint = manifest.get("artifacts", {}).get("checkpoint") if manifest else None
    if not checkpoint:
        continue
    command = [
        sys.executable,
        "scripts/export_model.py",
        "--config",
        str(project_config_path(item)),
        "--checkpoint",
        checkpoint,
        "--run-root",
        str(RUN_ROOT),
        "--skip-onnx",
    ]
    if RUN_EXPORT_PT:
        run_cli(command)
    else:
        print(" ".join(str(part) for part in command))


## 9. Run Manifest Snapshot

This table is only a quick status check. Use `food101_CNN_final_project.ipynb` after all chosen runs are complete to produce the meaningful comparison and final model selection.

In [ ]:
manifests = list_run_manifests(PROJECT_ROOT, run_root=RUN_ROOT)
rows = [manifest_to_comparison_row(item) for item in manifests]
pd.DataFrame(rows, columns=["model", "version", "trained_at", "checkpoint", "top1", "top5", "macro_f1", "ece", "latency_ms", "params", "notes"])